In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from reserve_ampleness.data.registry import resolve

data0 = json.loads(resolve("nyfed_all_rates").read_text())

#data0 = pd.read_json('../DATA/RAW/nyfed_rates_all_2000-01-01_2026-08-13_pulled20260813T172932Z.json')

In [ ]:
data1 = pd.json_normalize(data0['refRates'])

In [ ]:
data1['effectiveDate'] = pd.to_datetime(data1['effectiveDate'])

In [ ]:
data1[data1['revisionIndicator'].str.len() > 0]

In [ ]:
data1.dtypes

In [ ]:
data1.columns

In [ ]:
cols = ['average30day', 'average90day',
       'average180day', 'index', 'percentRate',
       'percentPercentile1', 'percentPercentile25', 'percentPercentile75',
       'percentPercentile99', 'targetRateFrom', 'targetRateTo',
       'volumeInBillions', 'footnoteId', 'intraDayLow', 'intraDayHigh',
       'stdDeviation']

In [ ]:
data1[data1['percentPercentile99'].str.len() > 0]

In [ ]:
data1[cols] = data1[cols].apply(pd.to_numeric, errors = 'coerce')

In [ ]:
data1.dtypes

In [ ]:
data2 = data1.set_index('effectiveDate')

In [ ]:
sofr = data2[data2['type'] == 'SOFR']
tgcr = data2[data2['type'] == 'TGCR']
bgcr = data2[data2['type'] == 'BGCR']
effr = data2[data2['type'] == 'EFFR']
ofbr = data2[data2['type'] == 'OBFR']
sofrai = data2[data2['type'] == 'SOFRAI']

In [ ]:
dfs = [sofr.add_prefix('SOFR'), bgcr.add_prefix('BGCR'), tgcr.add_prefix('TGCR')]
result_outer = pd.concat(dfs, axis = 1)

In [ ]:
len(result_outer[result_outer['SOFRpercentRate'] < result_outer['BGCRpercentRate']]), len(result_outer[result_outer['BGCRpercentRate'] < result_outer['TGCRpercentRate']]), len(result_outer[result_outer['SOFRpercentRate'] < result_outer['TGCRpercentRate']])

In [ ]:
condition = (
    (result_outer['SOFRpercentPercentile1'] <= result_outer['SOFRpercentPercentile25'])&
    (result_outer['SOFRpercentPercentile25'] <= result_outer['SOFRpercentRate'])&
    (result_outer['SOFRpercentRate'] <= result_outer['SOFRpercentPercentile75'])&
    (result_outer['SOFRpercentPercentile75'] <= result_outer['SOFRpercentPercentile99'])
)

result_outer[condition]

In [ ]:
result_outer[~condition][['SOFRpercentRate', 'BGCRpercentRate', 'TGCRpercentRate']]

In [ ]:
volcond = (
    (result_outer['SOFRvolumeInBillions'] >= result_outer['BGCRvolumeInBillions'])&
    (result_outer['BGCRvolumeInBillions'] >= result_outer['TGCRvolumeInBillions'])
)

In [ ]:
result_outer[~volcond]